# EC ENGR 279AS — RF/Microwave & Photonics Math Toolkit
### Dirac delta · Trig substitution · ε μ · Logarithms · 64-bit · QM↔EM

UCLA course context: physical and wave electronics, electromagnetics, photonics (Jalali group),  
plasma electronics, microwave/mm-wave circuits.  
Every section: **SymPy loops → NumPy vectorize → Torch batch**.

In [ ]:
import sympy as sp
import numpy as np
import torch
import matplotlib.pyplot as plt
import struct, math
from IPython.display import display, Math

sp.init_printing(use_latex='mathjax')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 110})
print('ready')

---
# §1 — Dirac Delta: Every Representation

**Plain English:** $\delta(x)$ is not a function — it is a *distribution*.  
It equals zero everywhere except $x=0$, and integrates to 1.  
In physics and engineering you meet it as the limit of many different peaked functions.  
Knowing all the representations lets you swap in whichever one makes your integral tractable.

$$\delta(x) = \lim_{\epsilon\to 0} D_\epsilon(x) \qquad \int_{-\infty}^{\infty}\delta(x)\,dx = 1$$

In [ ]:
x, eps, t, f_var, a_var, n_var = sp.symbols('x epsilon t f a n', real=True, positive=True)
x_s = sp.Symbol('x', real=True)

# ── all representations as (name, latex, numpy_fn) ────────────────────
dirac_reps = [
    ('Gaussian',
     r'\frac{1}{\epsilon\sqrt{\pi}}e^{-x^2/\epsilon^2}',
     lambda xv, e: np.exp(-xv**2/e**2) / (e * np.sqrt(np.pi))),

    ('Lorentzian',
     r'\frac{1}{\pi}\frac{\epsilon}{x^2+\epsilon^2}',
     lambda xv, e: (e/np.pi) / (xv**2 + e**2)),

    ('sinc²',
     r'\frac{1}{\pi x}\sin(x/\epsilon)',
     lambda xv, e: np.sinc(xv / (np.pi * e)) / e),   # np.sinc = sin(πx)/(πx)

    ('Top-hat',
     r'\frac{1}{2\epsilon}\mathbb{1}_{|x|<\epsilon}',
     lambda xv, e: np.where(np.abs(xv) < e, 1/(2*e), 0.0)),

    ('Fourier integral',
     r'\frac{1}{2\pi}\int_{-\infty}^{\infty}e^{ikx}dk',
     lambda xv, e: np.sinc(xv * e / np.pi) * e / np.pi),  # band-limited approx

    ('Derivative of Heaviside',
     r"\Theta'(x)",
     lambda xv, e: np.exp(-xv**2/(2*e**2)) / (e * np.sqrt(2*np.pi))),  # smooth approx
]

print('Dirac delta representations:\n')
for name, latex_str, _ in dirac_reps:
    display(Math(rf'\delta(x) \;[\text{{{name}}}] = \lim_{{\epsilon\to 0}} {latex_str}'))

# ── plot all reps at ε = 0.15 ─────────────────────────────────────────
xv   = np.linspace(-1.5, 1.5, 2000)
eps_val = 0.15

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, (name, _, fn) in zip(axes.flat, dirac_reps):
    y = fn(xv, eps_val)
    ax.plot(xv, y, color='royalblue', lw=1.8)
    ax.fill_between(xv, y, alpha=0.15, color='royalblue')
    norm = np.trapz(y, xv)
    ax.set_title(f'{name}\n∫dx ≈ {norm:.4f}', fontsize=10)
    ax.set_xlabel('x'); ax.set_xlim(-1, 1)
    ax.axvline(0, color='tomato', lw=0.8, ls='--')

plt.suptitle(rf'All $\delta(x)$ representations at $\epsilon={eps_val}$', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ── SymPy: key δ properties in a loop ────────────────────────────────
x_s = sp.Symbol('x', real=True)
a_s = sp.Symbol('a', real=True, nonzero=True)
f_s = sp.Function('f')

delta_properties = [
    ('Sifting',
     sp.Integral(f_s(x_s) * sp.DiracDelta(x_s - a_s), (x_s, -sp.oo, sp.oo)),
     f_s(a_s)),

    ('Scaling',
     sp.DiracDelta(a_s * x_s),
     sp.DiracDelta(x_s) / sp.Abs(a_s)),

    ('Even symmetry',
     sp.DiracDelta(-x_s),
     sp.DiracDelta(x_s)),

    ('Derivative sifting',
     sp.Integral(f_s(x_s) * sp.DiracDelta(x_s, 1), (x_s, -sp.oo, sp.oo)),
     -f_s(x_s).diff(x_s).subs(x_s, 0)),

    ('Convolution with f',
     sp.Symbol('(f * delta)(t)', real=True),
     sp.Symbol('f(t)', real=True)),
]

print('Dirac delta properties (SymPy loop):\n')
for name, lhs, rhs in delta_properties:
    display(Math(rf'\text{{{name}}}: \quad {sp.latex(lhs)} = {sp.latex(rhs)}'))

# ── Torch: numerical sifting demo ─────────────────────────────────────
# ∫ f(x) δ(x-a) dx ≈ f(a)  via Gaussian approx
a_val  = 0.73
eps_t  = 0.02
x_t    = torch.linspace(-5, 5, 100000, dtype=torch.float64)
dx     = (x_t[1] - x_t[0]).item()

f_t    = torch.sin(x_t) + 0.5 * x_t**2          # some function f(x)
delta_t = torch.exp(-(x_t - a_val)**2 / eps_t**2) / (eps_t * math.sqrt(math.pi))
integral = (f_t * delta_t * dx).sum().item()
f_at_a   = (math.sin(a_val) + 0.5 * a_val**2)

print(f'\nTorch sifting: ∫ f(x)δ(x−{a_val}) dx = {integral:.6f}')
print(f'Exact f({a_val})                       = {f_at_a:.6f}')
print(f'Error: {abs(integral - f_at_a):.2e}')

---
# §2 — Trig Substitution: All Three Cases

**Plain English:** when an integral contains $\sqrt{a^2-x^2}$, $\sqrt{a^2+x^2}$, or $\sqrt{x^2-a^2}$,  
you substitute a trig (or hyperbolic trig) function to kill the square root.  
These three cases cover *every* such integral in physics — Coulomb fields, waveguide cutoffs,  
relativistic momenta, optical path lengths.

| Integrand form | Substitution | Identity used |
|---|---|---|
| $\sqrt{a^2-x^2}$ | $x=a\sin\theta$ | $1-\sin^2=\cos^2$ |
| $\sqrt{a^2+x^2}$ | $x=a\tan\theta$ | $1+\tan^2=\sec^2$ |
| $\sqrt{x^2-a^2}$ | $x=a\sec\theta$ | $\sec^2-1=\tan^2$ |

In [ ]:
x_s, a_s, theta = sp.symbols('x a theta', positive=True)

# ── three canonical physics integrals, one per case ───────────────────
trig_cases = [
    # (case name, integrand, variable, substitution expr, physics context)
    ('Case 1: √(a²−x²)',
     1 / sp.sqrt(a_s**2 - x_s**2),
     x_s,
     a_s * sp.sin(theta),
     'Coulomb potential on a disk of radius a'),

    ('Case 2: √(a²+x²)',
     1 / (a_s**2 + x_s**2)**sp.Rational(3,2),
     x_s,
     a_s * sp.tan(theta),
     'E-field on axis of charged ring'),

    ('Case 3: √(x²−a²)',
     1 / (x_s * sp.sqrt(x_s**2 - a_s**2)),
     x_s,
     a_s / sp.cos(theta),
     'Waveguide evanescent field integral'),

    ('Case 1b: x²√(a²−x²)',
     x_s**2 * sp.sqrt(a_s**2 - x_s**2),
     x_s,
     a_s * sp.sin(theta),
     'Moment of inertia of a sphere'),

    ('Case 2b: √(1+x²)',
     sp.sqrt(1 + x_s**2),
     x_s,
     sp.tan(theta),
     'Arc length of a parabola / relativistic path'),

    ('Case 3b: 1/√(x²−1)',
     1 / sp.sqrt(x_s**2 - 1),
     x_s,
     1 / sp.cos(theta),
     'Inverse cosh — hyperbolic waveguide mode'),
]

print('Trig substitution — SymPy antiderivatives:\n')
for name, integrand, var, sub, context in trig_cases:
    result = sp.integrate(integrand, var)
    result_s = sp.simplify(result)
    display(Math(
        rf'\textbf{{{name}}} \;\; [{context}]'
    ))
    display(Math(
        rf'\int {sp.latex(integrand)}\,dx = {sp.latex(result_s)} + C'
    ))
    print()

# ── NumPy: verify numerically for Case 2 (ring E-field) ───────────────
a_val = 1.0
x_vals = np.linspace(-5, 5, 1000)
integrand_np = 1 / (a_val**2 + x_vals**2)**1.5

# analytical:  ∫₋∞^∞ dx/(a²+x²)^(3/2) = 2/a²
numerical = np.trapz(integrand_np, x_vals)
analytical = 2 / a_val**2
print(f'Case 2 definite integral (−∞ to ∞):  numerical={numerical:.6f},  analytical={analytical:.6f}')

In [ ]:
# ── visualize the three substitutions geometrically ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
theta_v = np.linspace(0, np.pi/2 - 0.05, 300)
a_v = 1.0

# Case 1: x = a sin θ  →  √(a²-x²) = a cos θ
ax = axes[0]
x1 = a_v * np.sin(theta_v)
y1 = a_v * np.cos(theta_v)   # √(a²-x²)
ax.plot(x1, y1, color='royalblue', lw=2)
ax.fill_between(x1, y1, alpha=0.15, color='royalblue')
ax.set_xlabel(r'$x = a\sin\theta$'); ax.set_ylabel(r'$\sqrt{a^2-x^2}$')
ax.set_title('Case 1: circle arc\n$x=a\\sin\\theta$'); ax.set_aspect('equal')

# Case 2: x = a tan θ  →  √(a²+x²) = a sec θ
ax = axes[1]
x2 = a_v * np.tan(theta_v)
y2 = a_v / np.cos(theta_v)   # a sec θ
ax.plot(x2, y2, color='seagreen', lw=2)
ax.set_xlabel(r'$x = a\tan\theta$'); ax.set_ylabel(r'$\sqrt{a^2+x^2}$')
ax.set_title('Case 2: hyperbola\n$x=a\\tan\\theta$')
ax.set_xlim(0, 5); ax.set_ylim(0, 6)

# Case 3: x = a sec θ  →  √(x²-a²) = a tan θ
ax = axes[2]
x3 = a_v / np.cos(theta_v)
y3 = a_v * np.tan(theta_v)   # a tan θ
ax.plot(x3, y3, color='tomato', lw=2)
ax.set_xlabel(r'$x = a\sec\theta$'); ax.set_ylabel(r'$\sqrt{x^2-a^2}$')
ax.set_title('Case 3: outer hyperbola\n$x=a\\sec\\theta$')
ax.set_xlim(0.9, 5); ax.set_ylim(0, 5)

plt.suptitle('Geometric meaning of the three trig substitution cases', fontsize=12)
plt.tight_layout(); plt.show()

---
# §3 — Permittivity ε, Permeability μ, and the Jalali Photonics Connection

**Plain English:**  
- $\varepsilon$ (permittivity) tells you how easily electric fields polarize a medium.  
- $\mu$ (permeability) tells you how easily magnetic fields magnetize it.  
- Together they set $c = 1/\sqrt{\varepsilon\mu}$, the wave speed in that medium.  
- The **refractive index** $n = \sqrt{\varepsilon_r\mu_r}$ and **impedance** $Z = \sqrt{\mu/\varepsilon}$  
  control reflection, transmission, dispersion, and group delay — the core of 279AS.  

**Jalali group context:** the dispersive Fourier transform (DFT) stretches a short pulse in a  
dispersive fiber so that time maps to frequency — $\varepsilon(\omega)$ (via GVD $\beta_2$) is the engine.

In [ ]:
omega_s, omega0, gamma_s, eps_inf = sp.symbols(
    'omega omega_0 gamma epsilon_infty', positive=True
)

# ── Drude-Lorentz permittivity ─────────────────────────────────────────
# ε(ω) = ε_∞ + (ε_s - ε_∞)ω₀² / (ω₀² - ω² + iγω)
eps_s_sym = sp.Symbol('epsilon_s', positive=True)
i_s       = sp.I

eps_Lorentz = eps_inf + (eps_s_sym - eps_inf) * omega0**2 / (
    omega0**2 - omega_s**2 + i_s * gamma_s * omega_s
)
display(Math(rf'\varepsilon(\omega) = {sp.latex(eps_Lorentz)}'))

# ── derived quantities via SymPy ───────────────────────────────────────
c_sym, mu_sym = sp.symbols('c mu', positive=True)
n_sym  = sp.sqrt(eps_Lorentz * mu_sym)   # general n
Z_sym  = sp.sqrt(mu_sym / eps_Lorentz)   # impedance

derived = [
    ('wave speed in medium',  r'v = c/n = c/\sqrt{\varepsilon_r\mu_r}'),
    ('refractive index',      r'n(\omega) = \sqrt{\varepsilon_r(\omega)\mu_r(\omega)}'),
    ('wave impedance',        r'Z(\omega) = \sqrt{\mu(\omega)/\varepsilon(\omega)}'),
    ('reflection coeff',      r'\Gamma = (Z_2 - Z_1)/(Z_2 + Z_1)'),
    ('phase velocity',        r'v_p = \omega/k = c/n(\omega)'),
    ('group velocity',        r'v_g = (d\omega/dk)^{-1}\cdot 1 = c\,/\,(n+\omega\,dn/d\omega)'),
    ('GVD (Jalali DFT)',      r'\beta_2 = d^2k/d\omega^2 = -(1/v_g^2)\,dv_g/d\omega'),
]
print('\nEM derived quantities:\n')
for name, formula in derived:
    display(Math(rf'\text{{{name}}}: \quad {formula}'))

# ── NumPy: Drude-Lorentz ε(ω), n(ω), Z(ω) curves ─────────────────────
omega_v = np.linspace(0.1, 3.0, 2000)   # normalized to ω₀

# parameters: silicon-like
eps_inf_v = 1.0; eps_s_v = 11.7; omega0_v = 1.0; gamma_v = 0.05

eps_v = eps_inf_v + (eps_s_v - eps_inf_v) * omega0_v**2 / (
    omega0_v**2 - omega_v**2 + 1j * gamma_v * omega_v
)
n_v   = np.sqrt(eps_v)         # complex refractive index
Z_v   = 1 / np.sqrt(eps_v)     # normalized impedance (μ_r=1)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (qty, re, im, title, unit) in zip(axes, [
    ('eps', eps_v.real, eps_v.imag, r"$\varepsilon_r(\omega)$", 'ε_r'),
    ('n',   n_v.real,   n_v.imag,   r"$n(\omega)$",             'n'),
    ('Z',   Z_v.real,   Z_v.imag,   r"$Z(\omega)/Z_0$",         'Z/Z₀'),
]):
    ax.plot(omega_v, re, color='royalblue', lw=2, label="Re")
    ax.plot(omega_v, im, color='tomato',    lw=2, label="Im", ls='--')
    ax.axvline(1.0, color='k', lw=0.8, ls=':')
    ax.set_xlabel(r'$\omega/\omega_0$'); ax.set_ylabel(unit)
    ax.set_title(title); ax.legend(); ax.set_ylim(-5, 15)

plt.suptitle('Drude-Lorentz: ε(ω), n(ω), Z(ω)  — resonance at ω=ω₀', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# ── Jalali dispersive Fourier transform (DFT) ──────────────────────────
# A short pulse in a dispersive fiber of length L with GVD β₂:
# output envelope ∝ FT of input, time axis = ω · β₂ · L

# parameters
beta2 = -20e-27       # s²/m  (anomalous dispersion, silica at 1550 nm)
L     = 10e3          # 10 km fiber
T0    = 1e-12         # 1 ps input pulse width

t_vals = np.linspace(-10e-12, 10e-12, 4000)  # time axis

# Gaussian input pulse
pulse_in = np.exp(-t_vals**2 / (2*T0**2))

# DFT output: time maps to frequency  t ↔ ω·β₂·L
# |E_out(t)|² ≈ |Ẽ_in(t / (β₂·L))|²   (up to phase)
omega_dft = t_vals / (beta2 * L)             # frequency mapped from time
freq_dft  = omega_dft / (2 * np.pi)

# stretched pulse (chirped Gaussian)
T_stretch = np.sqrt(T0**2 + (beta2 * L / T0)**2)
pulse_out = np.exp(-t_vals**2 / (2 * T_stretch**2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(t_vals*1e12, pulse_in,  color='royalblue', lw=2, label='input  (1 ps)')
axes[0].plot(t_vals*1e12, pulse_out, color='tomato',    lw=2, label=f'output ({T_stretch*1e12:.1f} ps)')
axes[0].set_xlabel('time (ps)'); axes[0].set_ylabel('amplitude')
axes[0].set_title('Jalali DFT: pulse stretching in dispersive fiber')
axes[0].legend()

axes[1].plot(freq_dft*1e-12, pulse_out, color='seagreen', lw=2)
axes[1].set_xlabel('frequency (THz)  [mapped from time]')
axes[1].set_ylabel('|E|')
axes[1].set_title(rf'Time→Frequency mapping: $f = t/(\beta_2 L)$, $\beta_2$={beta2*1e27:.0f} ps²/km')
axes[1].set_xlim(-2, 2)

plt.suptitle('Dispersive Fourier Transform (Jalali group, UCLA)', fontsize=12)
plt.tight_layout(); plt.show()

print(f'Stretch factor: T_out/T_in = {T_stretch/T0:.1f}x')
print(f'GVD length: L_D = T₀²/|β₂| = {T0**2/abs(beta2)/1e3:.2f} km')

---
# §4 — QM ↔ EM Correspondence

**Plain English:** quantum mechanics and classical EM share the same mathematical skeleton.  
The Schrödinger equation and the paraxial wave equation are the *same PDE* in different variables.  
Knowing this mapping lets you borrow solutions from one domain into the other.

| EM quantity | QM quantity | Shared math |
|---|---|---|
| $E_x(z,t)$ envelope | $\psi(x,t)$ wavefunction | same paraxial/Schrödinger PDE |
| refractive index $n(x)$ | potential $V(x)$ | same eigenvalue problem |
| group velocity $v_g$ | $\langle p\rangle/m$ | same dispersion relation |
| GVD $\beta_2$ | $\hbar^2/2m$ | same diffraction/dispersion coefficient |
| waveguide mode | bound state | same boundary condition |

In [ ]:
z_s, t_s, x_s2 = sp.symbols('z t x', real=True)
k_s, n_s, beta0, beta2_s = sp.symbols('k n beta_0 beta_2', real=True, positive=True)
hbar_s, m_s, V_s = sp.symbols('hbar m V', real=True, positive=True)

# ── the two PDEs side by side ──────────────────────────────────────────
pdes = [
    ('Paraxial wave eq (EM)',
     r'i\,\frac{\partial A}{\partial z} = -\frac{\beta_2}{2}\frac{\partial^2 A}{\partial t^2}'),
    ('Schrödinger eq (QM)',
     r'i\hbar\,\frac{\partial\psi}{\partial t} = -\frac{\hbar^2}{2m}\frac{\partial^2\psi}{\partial x^2} + V\psi'),
    ('Mapping',
     r'z\leftrightarrow t,\quad t\leftrightarrow x,\quad \beta_2\leftrightarrow\hbar/m,\quad n(x)\leftrightarrow V(x)'),
]
for name, eq in pdes:
    display(Math(rf'\text{{{name}}}: \quad {eq}'))

print()

# ── loop: compute eigenvalues for both PIB and dielectric waveguide ────
print('Eigenvalue comparison: particle-in-box vs slab waveguide\n')
for n_mode in range(1, 6):
    # QM: E_n = n²π²ħ²/(2mL²)
    E_qm = n_mode**2 * np.pi**2   # normalized: ħ=m=L=1
    # EM: k_z,n = √(k²n²_core - (nπ/d)²)  — cutoff when k_z=0  →  k_c = nπ/(d·n_core)
    k_cutoff = n_mode * np.pi     # normalized: d=1, n_core=1
    display(Math(
        rf'n={n_mode}: \quad E_{{\text{{QM}}}} = {E_qm:.3f}\hbar^2/2mL^2 '
        rf'\longleftrightarrow k_{{c,{n_mode}}} = {k_cutoff:.3f}/d'
    ))

# ── Torch: compute wavefunctions / modes simultaneously ───────────────
x_t = torch.linspace(0, 1, 500, dtype=torch.float64)
n_modes = torch.arange(1, 7, dtype=torch.float64)   # modes 1..6

# PIB wavefunctions ψ_n(x) = √2 sin(nπx)
psi_t = torch.sqrt(torch.tensor(2.0)) * torch.sin(
    torch.outer(n_modes * torch.pi, x_t)  # (6, 500)
)

fig, ax = plt.subplots(figsize=(9, 5))
colors = plt.cm.viridis(np.linspace(0.1, 0.9, 6))
E_n = (n_modes**2 * np.pi**2).numpy()
for i in range(6):
    ax.plot(x_t.numpy(), psi_t[i].numpy() + E_n[i],
            color=colors[i], lw=1.8, label=f'n={i+1}')
    ax.axhline(E_n[i], color=colors[i], lw=0.5, ls='--', alpha=0.5)

ax.set_xlabel('x/L'); ax.set_ylabel(r'$E_n + \psi_n(x)$  [normalized]')
ax.set_title('PIB wavefunctions (QM) = slab waveguide modes (EM)\nTorch batch computation')
ax.legend(ncol=2, fontsize=9); plt.tight_layout(); plt.show()

---
# §5 — 64-bit IEEE 754 Representation + Photonic Analogy

**Plain English:**  
A 64-bit `double` = 1 sign bit + 11 exponent bits + 52 mantissa bits.  
The exponent is **biased**: stored value = true exponent + 1023.  
Photonic analogy: in wavelength-division multiplexing (WDM), each `channel = 1 bit position`,  
the carrier frequency = exponent, the amplitude = mantissa.

In [ ]:
import struct

def float64_breakdown(val):
    """Return (sign, exponent_biased, exponent_true, mantissa_bits, mantissa_val)."""
    packed = struct.pack('>d', val)
    bits   = int.from_bytes(packed, 'big')
    sign   = (bits >> 63) & 1
    exp_b  = (bits >> 52) & 0x7FF
    mant   = bits & ((1 << 52) - 1)
    exp_t  = exp_b - 1023
    mant_v = 1.0 + mant / (2**52)   # implicit leading 1
    return sign, exp_b, exp_t, f'{mant:052b}', mant_v

# ── loop over physics-relevant values ─────────────────────────────────
physics_vals = [
    ('c  (speed of light)',     2.998e8),
    ('e  (electron charge)',    1.602e-19),
    ('ħ  (reduced Planck)',     1.055e-34),
    ('ε₀ (permittivity)',       8.854e-12),
    ('π',                       math.pi),
    ('1.0',                     1.0),
    ('0.1  (not exact in bin)', 0.1),
]

print(f'{"Value":28} {"sign":5} {"exp(biased)":12} {"exp(true)":10} {"mantissa (first 20 bits)"}')
print('-'*95)
for name, val in physics_vals:
    s, eb, et, mb, mv = float64_breakdown(val)
    print(f'{name:28} {s:5d} {eb:12d} {et:10d}   {mb[:20]}...')

# ── visualize 64-bit layout for π ────────────────────────────────────
val_show = math.pi
s, eb, et, mb, mv = float64_breakdown(val_show)
bits_str = f'{s}' + f'{eb:011b}' + mb

fig, ax = plt.subplots(figsize=(14, 2.5))
colors_bits = ['tomato']*1 + ['seagreen']*11 + ['royalblue']*52
for i, (b, col) in enumerate(zip(bits_str, colors_bits)):
    ax.add_patch(plt.Rectangle((i, 0), 0.95, 1, color=col, alpha=0.7))
    ax.text(i+0.47, 0.5, b, ha='center', va='center', fontsize=7, fontweight='bold')

ax.add_patch(plt.Rectangle((0, 1.1), 1, 0.3, color='tomato',   alpha=0.5))
ax.text(0.5, 1.25, 'sign', ha='center', fontsize=8)
ax.add_patch(plt.Rectangle((1, 1.1), 11, 0.3, color='seagreen', alpha=0.5))
ax.text(6.5, 1.25, 'exponent (11 bits, bias=1023)', ha='center', fontsize=8)
ax.add_patch(plt.Rectangle((12, 1.1), 52, 0.3, color='royalblue', alpha=0.5))
ax.text(38, 1.25, 'mantissa (52 bits, implicit leading 1)', ha='center', fontsize=8)

ax.set_xlim(-0.5, 64.5); ax.set_ylim(-0.2, 1.6)
ax.axis('off')
ax.set_title(rf'IEEE 754 double: $\pi$ = {val_show}   → sign={s}, exp_true={et}, mantissa≈{mv:.15f}')
plt.tight_layout(); plt.show()

# ── photonic WDM analogy ───────────────────────────────────────────────
print('\nPhotonic WDM analogy:')
print('  64 channels = 64 wavelengths in a WDM comb')
print('  Channel 63 (MSB) = sign  → carrier phase flip')
print('  Channels 52-62   = exponent → carrier frequency range')
print('  Channels 0-51    = mantissa → amplitude modulation depth')

# WDM channel spacing (C-band, 50 GHz ITU grid)
f_start = 193.1e12   # Hz
df      = 50e9
wdm_freqs = f_start + np.arange(64) * df
wdm_lambda = 3e8 / wdm_freqs * 1e9  # nm
print(f'\n  64-channel WDM: λ = {wdm_lambda[0]:.2f} nm to {wdm_lambda[-1]:.2f} nm  (50 GHz spacing)')

---
# §6 — Heavy Logarithms in Physics

**Plain English:** logarithms are everywhere in RF and photonics because:  
- Power in circuits varies over 12+ orders of magnitude → **dB scale**  
- Entropy, information, and quantum states → **natural log**  
- Impedance matching, S-parameters, noise figure → **log ratios**  
- Electromagnetic skin depth, waveguide attenuation → **exponential decay → log**

In [ ]:
P_s, P0_s, f_s2, sigma_s, T_s = sp.symbols('P P_0 f sigma T', positive=True)
Z1_s, Z2_s, R_s = sp.symbols('Z_1 Z_2 R', positive=True)
k_B_s = sp.Symbol('k_B', positive=True)

# ── all the log formulas in physics/RF/photonics ──────────────────────
log_formulas = [
    ('Power dB',             r'P_{\text{dB}} = 10\log_{10}(P/P_0)'),
    ('Amplitude dB',         r'A_{\text{dB}} = 20\log_{10}(A/A_0)'),
    ('Noise figure',         r'NF = 10\log_{10}(F),\quad F=(S/N)_{\text{in}}/(S/N)_{\text{out}}'),
    ('Shannon capacity',     r'C = B\log_2(1 + S/N)  \text{ [bits/s]}'),
    ('Boltzmann entropy',    r'S = k_B\ln\Omega'),
    ('von Neumann entropy',  r'S = -k_B\,\text{Tr}(\rho\ln\rho)'),
    ('Friis transmission',   r'P_r = P_t G_t G_r(\lambda/4\pi d)^2'),
    ('Path loss',            r'L_{\text{dB}} = 20\log_{10}(4\pi d/\lambda)'),
    ('Skin depth',           r'\delta = \sqrt{2/(\omega\mu\sigma)},\quad \alpha = 1/\delta'),
    ('Optical loss',         r'\alpha_{\text{dB/km}} = 10\log_{10}(e)\cdot 2\alpha_{\text{Np/m}}\cdot 1000'),
    ('Q factor',             r'Q = \omega_0 / \Delta\omega = -20\log_{10}(1/\sqrt{2})^{-1}'),
    ('Bode magnitude',       r'|H(j\omega)|_{\text{dB}} = 20\log_{10}|H|'),
]

print('Log formulas in RF / photonics / physics:\n')
for name, formula in log_formulas:
    display(Math(rf'\text{{{name}}}: \quad {formula}'))

# ── NumPy: plot dB scale power, Shannon capacity, skin depth ──────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# dB scale
P_ratio = np.logspace(-6, 3, 500)
axes[0].semilogx(P_ratio, 10*np.log10(P_ratio), color='royalblue', lw=2)
axes[0].axhline(0, color='k', lw=0.5); axes[0].axvline(1, color='k', lw=0.5)
axes[0].set_xlabel('P/P₀'); axes[0].set_ylabel('dB')
axes[0].set_title('Power dB scale')
for ref, label in [(0.5, '−3 dB'), (2, '+3 dB'), (10, '+10 dB')]:
    axes[0].annotate(label, xy=(ref, 10*np.log10(ref)),
                     xytext=(ref*2, 10*np.log10(ref)+3), fontsize=8,
                     arrowprops=dict(arrowstyle='->', lw=0.8))

# Shannon capacity
snr_db = np.linspace(-10, 40, 500)
snr    = 10**(snr_db/10)
C_norm = np.log2(1 + snr)   # bits/s/Hz
axes[1].plot(snr_db, C_norm, color='seagreen', lw=2)
axes[1].set_xlabel('SNR (dB)'); axes[1].set_ylabel('C/B  (bits/s/Hz)')
axes[1].set_title('Shannon capacity')

# Skin depth vs frequency (copper)
f_hz   = np.logspace(3, 12, 500)   # 1 kHz to 1 THz
mu_Cu  = 4*np.pi*1e-7
sig_Cu = 5.96e7
delta  = np.sqrt(2 / (2*np.pi*f_hz * mu_Cu * sig_Cu))
axes[2].loglog(f_hz/1e9, delta*1e6, color='tomato', lw=2)
axes[2].set_xlabel('f (GHz)'); axes[2].set_ylabel('δ (μm)')
axes[2].set_title('Skin depth in copper\n(log-log: slope = −1/2)')

plt.suptitle('Logarithms in RF and photonics', fontsize=12)
plt.tight_layout(); plt.show()

# ── Torch: batch SNR → capacity ───────────────────────────────────────
snr_t = torch.tensor(snr, dtype=torch.float64)
C_t   = torch.log2(1 + snr_t)
err   = (C_t.numpy() - C_norm).max()
print(f'Torch vs NumPy Shannon capacity max err: {err:.2e}')

---
# §7 — Four-vector p·q, Relativistic Invariants

**Plain English:** in special relativity, the 4-momentum $p^\mu = (E/c, \mathbf{p})$.  
The inner product $p\cdot q = p^\mu q_\mu = E_p E_q/c^2 - \mathbf{p}\cdot\mathbf{q}$  
is **Lorentz invariant** — same value in every frame.  
This is used constantly in particle physics and in relativistic plasma / RF theory.

In [ ]:
Ep, Eq = sp.symbols('E_p E_q', positive=True)
px, py, pz = sp.symbols('p_x p_y p_z', real=True)
qx, qy, qz = sp.symbols('q_x q_y q_z', real=True)
c_s2 = sp.Symbol('c', positive=True)

# metric signature (+,-,-,-)
p4 = sp.Matrix([Ep/c_s2, px, py, pz])
q4 = sp.Matrix([Eq/c_s2, qx, qy, qz])
eta = sp.diag(1, -1, -1, -1)   # Minkowski metric

pq = (p4.T * eta * q4)[0,0]
pq_s = sp.simplify(pq)
display(Math(rf'p\cdot q = p^\mu\eta_{{\mu\nu}}q^\nu = {sp.latex(pq_s)}'))

# ── loop: compute p·p for each standard particle ──────────────────────
particles = [
    ('electron',  9.109e-31, 'massive'),
    ('proton',    1.673e-27, 'massive'),
    ('photon',    0.0,       'massless'),
    ('neutrino',  1e-36,     'nearly massless'),
]

c_val = 2.998e8
print('\nRest-frame invariant  p·p = (mc)²:\n')
for name, m, kind in particles:
    mc2 = (m * c_val)**2
    display(Math(rf'\text{{{name}}} \;({kind}): \quad p\cdot p = (mc)^2 = {mc2:.3e}\;\text{{kg}}^2\text{{m}}^2\text{{s}}^{{-2}}'))

# ── Torch: batch Lorentz inner products ──────────────────────────────
torch.manual_seed(0)
N = 10000
# random massive particles at rest-ish: p = (mc, p_x, p_y, p_z)
m_val = 9.109e-31
p_spatial = torch.randn(N, 3, dtype=torch.float64) * m_val * c_val * 0.1
E_vals_t  = torch.sqrt((m_val*c_val)**2 + (p_spatial**2).sum(dim=1)) * c_val
p4_t      = torch.cat([E_vals_t.unsqueeze(1)/c_val, p_spatial], dim=1)  # (N,4)

# p·p = E²/c² - |p|²  should = (mc)² for all
metric_t  = torch.diag(torch.tensor([1.,-1.,-1.,-1.], dtype=torch.float64))
pp_t      = (p4_t @ metric_t * p4_t).sum(dim=1)   # (N,)
expected  = (m_val * c_val)**2
err_pp    = (pp_t - expected).abs().max().item()
print(f'\nTorch: max |p·p − (mc)²| over {N} events = {err_pp:.3e}  ✓')

---
# §8 — Constant Power, S-parameters, and RF Transmitter Design

**Plain English:** in an RF transmitter (279AS core topic), you want **constant output power**  
across frequency. This requires understanding:  
- $S_{21}$ (transmission) vs frequency  
- Conjugate matching for max power transfer  
- How $\varepsilon(\omega)$ and $\mu(\omega)$ create frequency-dependent impedance  
- The relationship between analog bandwidth and discrete-time sampling (Nyquist)

In [ ]:
omega_rf, Z0_s, ZL_s, RL_s, CL_s = sp.symbols(
    'omega Z_0 Z_L R_L C_L', positive=True
)

# ── S11 for RC load on Z0 line ─────────────────────────────────────────
ZL_RC = RL_s / (1 + sp.I * omega_rf * RL_s * CL_s)
S11   = (ZL_RC - Z0_s) / (ZL_RC + Z0_s)
S11_s = sp.simplify(S11)
display(Math(rf'S_{{11}} = \frac{{Z_L - Z_0}}{{Z_L + Z_0}} = {sp.latex(S11_s)}'))

# power delivered to load: P = (1 - |S11|²) · P_avs
S11_mag2 = sp.simplify(sp.Abs(S11_s)**2)
display(Math(rf'|S_{{11}}|^2 = {sp.latex(S11_mag2)}'))
display(Math(r'P_{\text{delivered}} = (1 - |S_{11}|^2)\cdot P_{\text{avs}}'))

# ── NumPy: constant-power transmitter analysis ────────────────────────
f_hz2  = np.logspace(6, 12, 2000)   # 1 MHz to 1 THz
omega2 = 2*np.pi*f_hz2
Z0_v   = 50.0
RL_v   = 50.0
CL_v   = 1e-12   # 1 pF

ZL_v   = RL_v / (1 + 1j*omega2*RL_v*CL_v)
S11_v  = (ZL_v - Z0_v) / (ZL_v + Z0_v)
P_del  = 1 - np.abs(S11_v)**2   # normalized to available power

# conjugate match frequency: Im(ZL) = 0  → always matched for purely real ZL
# but RC load: ZL(ω) becomes complex → mismatch grows at high ω
f_3dB  = 1 / (2*np.pi*RL_v*CL_v)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].semilogx(f_hz2/1e9, 10*np.log10(np.abs(S11_v)**2 + 1e-20), color='tomato', lw=2)
axes[0].axvline(f_3dB/1e9, color='k', ls='--', lw=1)
axes[0].set_xlabel('f (GHz)'); axes[0].set_ylabel('|S₁₁|² (dB)')
axes[0].set_title('Return loss vs frequency')
axes[0].set_ylim(-40, 0)

axes[1].semilogx(f_hz2/1e9, 10*np.log10(P_del + 1e-20), color='royalblue', lw=2)
axes[1].axvline(f_3dB/1e9, color='k', ls='--', lw=1, label=f'f₃dB={f_3dB/1e9:.2f} GHz')
axes[1].set_xlabel('f (GHz)'); axes[1].set_ylabel('P_delivered / P_avs (dB)')
axes[1].set_title('Delivered power (constant-P transmitter)'); axes[1].legend()
axes[1].set_ylim(-10, 0.5)

axes[2].semilogx(f_hz2/1e9, np.real(ZL_v), color='seagreen', lw=2, label='Re(ZL)')
axes[2].semilogx(f_hz2/1e9, np.imag(ZL_v), color='tomato', lw=2, ls='--', label='Im(ZL)')
axes[2].axhline(Z0_v, color='k', lw=0.8, ls=':', label='Z₀=50Ω')
axes[2].set_xlabel('f (GHz)'); axes[2].set_ylabel('Z_L (Ω)')
axes[2].set_title('RC load impedance'); axes[2].legend(fontsize=8)

plt.suptitle('EC ENGR 279AS: RF transmitter matching — RC load on 50Ω line', fontsize=11)
plt.tight_layout(); plt.show()

# ── Torch: batch over load values ─────────────────────────────────────
C_vals = torch.logspace(-14, -9, 50, dtype=torch.float64)  # 10 fF to 1 nF
omega_t = torch.tensor(2*np.pi*10e9, dtype=torch.float64)  # 10 GHz
ZL_t   = RL_v / (1 + 1j * omega_t * RL_v * C_vals)
S11_t  = (ZL_t - Z0_v) / (ZL_t + Z0_v)
P_t    = 1 - S11_t.abs()**2

plt.figure(figsize=(6, 3))
plt.semilogx(C_vals.numpy()*1e12, (10*torch.log10(P_t + 1e-20)).numpy(),
             color='royalblue', lw=2)
plt.xlabel('C_L (pF)'); plt.ylabel('P_del (dB)')
plt.title('Torch batch: delivered power at 10 GHz vs C_L')
plt.axhline(-3, color='tomato', ls='--', label='−3 dB'); plt.legend()
plt.tight_layout(); plt.show()

---
# Summary — The 279AS Math Toolkit

| Section | Core tool | Key result |
|---------|----------|------------|
| §1 Dirac delta | SymPy `DiracDelta`, 6 representations | sifting property, Torch numerical verify |
| §2 Trig sub | `sp.integrate`, 6 cases | closed-form antiderivatives, NumPy verify |
| §3 ε, μ | Drude-Lorentz, `sp.symbols` | n(ω), Z(ω), Jalali DFT pulse stretch |
| §4 QM↔EM | Schrödinger = paraxial wave | PIB = slab modes, Torch batch wavefunctions |
| §5 64-bit | `struct.pack`, IEEE 754 loop | WDM photonic analogy |
| §6 Logs | dB, Shannon, skin depth | Torch Shannon capacity batch |
| §7 p·q | Minkowski metric, 4-vectors | Lorentz invariant verify in Torch |
| §8 RF power | S-parameters, conjugate match | constant-power transmitter, RC load |